# 07 — Kaggle Dataset Ablation Training (KAGGLE GPU)
## WavSent-MTL · Tasks 3.13–3.16

**Runs on: Kaggle T4 x2 GPU**

**What this notebook does:**
- Same ablation (Configs A–F) as notebook 06, but on Kaggle dataset
- Kaggle dataset has n_features=9 (adds polarity_max)
- BEST_REPR must already be set in config.py from notebook 06
- Saves: `kaggle_ablation_AG.csv` + val/test prediction arrays

**PREREQUISITE:**
- Notebook 06 complete
- BEST_REPR set in config.py and pushed to GitHub
- Upload `data/processed/kaggle/` as Kaggle dataset: `wavsent-kaggle-processed`

In [ ]:
# ── Kaggle Setup ────────────────────────────────────────────────────────────
import subprocess
subprocess.run(['git', 'clone', 'https://github.com/IAMNEERAJ05/WavSent-MTL.git',
                '/kaggle/working/WavSent-MTL'], check=True)

import sys
import os
sys.path.insert(0, '/kaggle/working/WavSent-MTL')
os.chdir('/kaggle/working/WavSent-MTL')

import torch
import gc
from config.config import CONFIG

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch: {torch.__version__}')
print(f'Device:  {device}')
if device == 'cuda':
    print(f'GPU:     {torch.cuda.get_device_name(0)}')
print(f'BEST_REPR: {CONFIG["BEST_REPR"]}')

print()
print('─' * 45)
print('Training Configuration')
print('─' * 45)
print(f"Early stopping patience : {CONFIG['early_stopping_patience']}")
print(f"Early stopping monitor  : {CONFIG['early_stopping_monitor']}")
print(f"Max epochs              : {CONFIG['max_epochs']}")
print(f"LR reduce patience      : {CONFIG['lr_reduce_patience']}")
print(f"LR reduce monitor       : {CONFIG['lr_reduce_monitor']}")
print(f"LR reduce factor        : {CONFIG['lr_reduce_factor']}")
print(f"Loss type               : {CONFIG['loss_type']}")
print('─' * 45)

## Load Kaggle Processed Arrays

In [ ]:
import numpy as np
import pandas as pd
import json

KAGGLE_INPUT = '/kaggle/input/wavsent-kaggle-processed'

# Full data dict (Config C/BEST selected features — 9 features)
data_C = {
    'X_train':      np.load(f'{KAGGLE_INPUT}/X_train.npy'),
    'X_val':        np.load(f'{KAGGLE_INPUT}/X_val.npy'),
    'X_test':       np.load(f'{KAGGLE_INPUT}/X_test.npy'),
    'y_clf_train':  np.load(f'{KAGGLE_INPUT}/y_clf_train.npy'),
    'y_clf_val':    np.load(f'{KAGGLE_INPUT}/y_clf_val.npy'),
    'y_clf_test':   np.load(f'{KAGGLE_INPUT}/y_clf_test.npy'),
    'y_reg_train':  np.load(f'{KAGGLE_INPUT}/y_reg_train.npy'),
    'y_reg_val':    np.load(f'{KAGGLE_INPUT}/y_reg_val.npy'),
    'y_reg_test':   np.load(f'{KAGGLE_INPUT}/y_reg_test.npy'),
}

with open(f'{KAGGLE_INPUT}/selected_features.json') as f:
    selected_features = json.load(f)   # 9 features

with open(f'{KAGGLE_INPUT}/class_weights.json') as f:
    class_weights = json.load(f)

print(f'X_train shape:     {data_C["X_train"].shape}')
print(f'Selected features: {selected_features}')
print(f'Class weights:     {class_weights}')

## Build Config A and B Data Variants

In [ ]:
from src.data.windows import create_windows, generate_targets, temporal_split
from src.data.preprocessor import apply_scaler, apply_reg_scaler

df_feat = pd.read_csv(f'{KAGGLE_INPUT}/featured_data.csv')
df_feat['Date'] = pd.to_datetime(df_feat['Date'])
df_feat = df_feat.sort_values('Date').reset_index(drop=True)

train_df, val_df, test_df = temporal_split(df_feat)
print(f'Split: Train={len(train_df)} Val={len(val_df)} Test={len(test_df)}')

def build_data_variant(feature_cols, train_df, val_df, test_df, w=5):
    tr_s, v_s, te_s, _ = apply_scaler(
        train_df[feature_cols].values,
        val_df[feature_cols].values,
        test_df[feature_cols].values,
        save_path='/tmp/tmp_scaler.pkl'
    )
    X_train = create_windows(tr_s, w)
    X_val   = create_windows(v_s, w)
    X_test  = create_windows(te_s, w)
    y_clf_tr, y_reg_tr = generate_targets(train_df['Close'].values, w)
    y_clf_v,  y_reg_v  = generate_targets(val_df['Close'].values, w)
    y_clf_te, y_reg_te = generate_targets(test_df['Close'].values, w)
    y_reg_tr, y_reg_v, y_reg_te, _ = apply_reg_scaler(
        y_reg_tr, y_reg_v, y_reg_te, save_path='/tmp/tmp_reg_scaler.pkl')
    return {
        'X_train': X_train, 'X_val': X_val, 'X_test': X_test,
        'y_clf_train': y_clf_tr, 'y_clf_val': y_clf_v, 'y_clf_test': y_clf_te,
        'y_reg_train': y_reg_tr, 'y_reg_val': y_reg_v, 'y_reg_test': y_reg_te,
    }

# Config A: return + polarity_mean + polarity_max (Kaggle has both)
for split_df in [train_df, val_df, test_df]:
    split_df['return'] = split_df['Close'].pct_change().fillna(0)

CONFIG_A_FEATURES = ['return', 'polarity_mean', 'polarity_max']
data_A = build_data_variant(CONFIG_A_FEATURES, train_df, val_df, test_df)
print(f'Config A X_train: {data_A["X_train"].shape}  (3 features)')

# Config B: denoised OHLCV (5) + polarity_mean + polarity_max = 7 features
CONFIG_B_FEATURES = ['Open_d', 'High_d', 'Low_d', 'Close_d', 'Volume_d',
                     'polarity_mean', 'polarity_max']
data_B = build_data_variant(CONFIG_B_FEATURES, train_df, val_df, test_df)
print(f'Config B X_train: {data_B["X_train"].shape}  (7 features)')
print(f'Config C X_train: {data_C["X_train"].shape}  (9 features — already loaded)')

## Helper: Run One Config

In [ ]:
from src.training.trainer import train_multi_run

RESULTS_PATH = '/kaggle/working/kaggle_ablation_partial.csv'
all_results = pd.DataFrame()


def run_config(config_name, model_name, data, class_weights=None):
    global all_results
    n_features = data['X_train'].shape[2]
    print(f'\n{"=" * 60}')
    print(f'Config {config_name} | {model_name.upper()} | n_features={n_features}')
    print(f'{"=" * 60}')

    results = train_multi_run(
        config_name=config_name,
        model_name=model_name,
        n_features=n_features,
        data=data,
        dataset='kaggle',
        class_weights=class_weights,
        device=device,
    )

    all_results = pd.concat([all_results, results], ignore_index=True)
    all_results.to_csv(RESULTS_PATH, index=False)

    mean_val  = results['val_accuracy'].mean()
    mean_test = results['accuracy'].mean()
    print(f'Config {config_name} | mean val_acc={mean_val:.4f} | mean test_acc={mean_test:.4f}')

    torch.cuda.empty_cache()
    gc.collect()
    return results


# Select BEST_REPR data
BEST_REPR = CONFIG['BEST_REPR']
data_BEST = data_B if BEST_REPR == 'denoised_ohlcv' else data_C
print(f'BEST_REPR: {BEST_REPR} | n_features={data_BEST["X_train"].shape[2]}')

## Run All Configs A–F

In [ ]:
results_A = run_config('A', 'tkan', data_A, class_weights)

In [ ]:
results_B = run_config('B', 'tkan', data_B, class_weights)

In [ ]:
results_C = run_config('C', 'tkan', data_C, class_weights)

In [ ]:
results_D = run_config('D', 'lstm', data_BEST, class_weights)

In [ ]:
results_E = run_config('E', 'gru', data_BEST, class_weights)

In [ ]:
results_F = run_config('F', 'tcn', data_BEST, class_weights)

## Summary

In [ ]:
final_path = '/kaggle/working/kaggle_ablation_AG.csv'
all_results.to_csv(final_path, index=False)

print('=' * 60)
print('Notebook 07 -- Kaggle Training: COMPLETE')
print('=' * 60)

summary = all_results.groupby('config').agg(
    mean_val_acc=('val_accuracy', 'mean'),
    mean_test_acc=('accuracy', 'mean'),
    std_test_acc=('accuracy', 'std'),
    mean_auc=('auc', 'mean'),
).round(4)
print(summary.to_string())

print(f'\nDownload from /kaggle/working/:')
print('  kaggle_ablation_AG.csv')
print()
print('Next: run notebook 08_ensemble')